# Particle in a Box

In this notebook, we will develop and explore a simple quantum mechanical model used to describe electrons confined within molecules. The system we study (a particle confined to a one-dimensional box) is one of the simplest quantum models, yet it captures essential features of real molecules (particularly conjugated polyenes). 

By implementing this model computationally, analyzing its predictions, and comparing them both to experiment and to more realistic quantum chemical calculations, you will investigate both the power and the limitations of theoretical models in chemistry. 

Throughout this notebook, the emphasis is on connecting equations, code, and physical interpretation so that each numerical result can be understood as a statement about molecular behavior.

---

**Chemistry Learning Objectives:**
- C4.1: Solve the Schrödinger equation for the particle in a box (PIB) using numerical methods.
- C4.2: Connect features of PIB wavefunctions to their energies.
- C4.3: Use the Self-Consistent Field/Hartree-Fock (SCF) method to predict chemical phenomena.
- C4.4: Compare predictions of PIB and SCF models for conjugated polyenes.

**Programming Learning Objectives:**
- P4.1: Perform numerical optimization routines with `scipy`.
- P4.2: Fit nonlinear curves to data using numerical optimization.
- P4.3: Perform quantum chemical calculations with the `psi4` software package.

---

**Table of Contents**
- [Warmup - Curve Fitting](#warmup) `20 points`
- [Part 1 - Solving the Schrödinger Equation](#part1) `40 points`
- [Part 2 - Energy Levels of the Particle in a Box](#part2) `30 points`
- [Part 3 - Computational Chemistry](#part3) `45 points`
- [Part 4 - Comparing Models](#part4) `30 points`
- [Reflection](#reflection) `10 points`

`Total: 175 points`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit, minimize
import psi4
import fortecubeview
import shutil

from pib_helper import (compute_energy,
                        load_psi4_molecule, calculate_r_squared)

<a id="warmup"></a>

## Warmup - Curve Fitting and Models

> *"All models are wrong, but some are useful."*

Throughout science, we use models to make sense of complex systems. No model captures reality perfectly and all models offer a tradeoff between simplicity and accuracy. In this lab, we will build several models of increasing sophistication and evaluate how well each one works.

A foundational skill for working with models is *curve fitting*: given experimental (or computational) data, finding a mathematical function that describes it and optimizing its parameters. Let's practice this now.

### Coding Activity (Warmup)
`15 points`

- P4.2: Fit nonlinear curves to data.

You have gotten some practice performing linear regression on data. You aren't limited to this - you can actually fit arbitrary functional forms. We'll use `scipy.optimize.curve_fit` for this.


#### W.A

Let's take a look at this mysterious data. 

**Your task:**
1. Load the data from `data/mystery_curve.csv` 
2. Make a scatter plot with axis labels and a title.


In [ ]:
# Subgoal: Load and inspect the data
mystery_curve = pd.read_csv('data/mystery_curve.csv')
plt.scatter(mystery_curve['x'], mystery_curve['y'], s=10)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Mystery Curve Data')
plt.show()

#### W.B - Learning from a worked example

Now we will practice fitting functions of our creation to this data! We'll use [`scipy.optimize.curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html) - read the linked documentation if you want more detail on how it works.

**Your task:**
1. Read the code in the two cells below carefully.
2. Add a comment to each line explaining what it does and why.
3. Run both cells and observe the output.

In [ ]:
from scipy.optimize import curve_fit

def abs_x(x, m, b):
    return m * np.abs(x) + b

params, covariance = curve_fit(
                    abs_x,                    # YOUR COMMENT HERE
                    mystery_curve['x'],       # YOUR COMMENT HERE
                    mystery_curve['y'],       # YOUR COMMENT HERE
                    p0=[1, 0])                # YOUR COMMENT HERE
print("Params:", params)

In [ ]:
plt.scatter(mystery_curve['x'], mystery_curve['y'], label='Data')  # YOUR COMMENT HERE

x_fit = np.linspace(-5, 5, 100)               # YOUR COMMENT HERE
m = params[0]                                  # YOUR COMMENT HERE
b = params[1]                                  # YOUR COMMENT HERE
y_fit = abs_x(x_fit, m, b)                    # YOUR COMMENT HERE
plt.plot(x_fit, y_fit, label='Fitted', color='red')  # YOUR COMMENT HERE

plt.xlabel('x')                                # YOUR COMMENT HERE
plt.ylabel('y')                                # YOUR COMMENT HERE
plt.title('Fitting Abs x Function')            # YOUR COMMENT HERE
plt.legend()                                   # YOUR COMMENT HERE
plt.show()

#### W.C - Fitting your own function

Now it's your turn! Define a different function and fit it to the same data.

**Your task:**
1. Define a new candidate function (e.g., Gaussian, exponential decay, quadratic - anything you think might match the shape of the data).
2. Use `curve_fit` to find the best parameters.
3. Plot the data and your fitted curve together.
4. Does your function fit the data better or worse than the absolute-value function above?

In [ ]:
# Subgoal: Define your candidate function
# # YOUR CODE HERE

# Subgoal: Fit your function to the data
# params, covariance = curve_fit(...)

# y_fit = your_function(x, *params) # * is the unpacking operator if used on a list

# Subgoal: Plot the data and your fitted curve
# plt.scatter(...)
# plt.plot(...)
# plt.xlabel(...)
# plt.ylabel(...)
# plt.title(...)
# plt.legend()
# plt.show()

### Question (Warmup)
`5 points`

a) Which of your candidate functions best fit the data? How did you decide?

b) Propose an example of a situation where you could use this technique in a real scientific application.

c) In the example you gave, how would the curve be useful to us? How about the parameters?

---

*Your answer here (`double click me!`):*

<br></br>

<a id="part1"></a>
## Part 1 - Numerically Solving the Schrodinger Equation

**The Hamiltonian**

Suppose we wanted to calculate the energy of an electron. What would we do?

Well, we can use the *Hamiltonian*, the operator that gives us the total energy of our wavefunction (as the sum of the potential and kinetic energies).
$$
\hat{H} \psi{}(x) = \hat{T}\psi{}(x)  + \hat{V}(x)\psi(x)
$$
Where $\psi{}(x)$ is the wavefunction, $\hat{H}$ is the Hamiltonian (i.e. the Energy operator), $\hat{T}$ is the kinetic energy operator, and $\hat{V}(x)$ is the potential energy (as a function of x).


**Solving the Schrodinger Equation**

For a wavefunction to have a clearly defined energy, it must satisfy the Time-Independent Schrodinger Equation: 
$$ \hat{H}\psi = E \psi $$
Where $E$ is constant. 

In general this is not true. The Hamiltonian involves multiplying $\psi$ by some arbitrary function of x, and adding this to its second derivative. There are very few functions that will come out of this unchanged!

In these special cases, we say that $\psi$ is an _eigenfunction_ of the energy operator $\hat{H}$, and the observed energy $E$ is an _eigenvalue_. 

Okay, great. If you want the wavefunction of a system, just solve the differential equation to find the eigenvalues of the Schrodinger Equation. And in fact, we can do that for the particle-in-a-box, as you may have learned in lecture. But what happens when we can't do this?

**Numerical Solutions**

Most real chemical systems, however, do *not* admit closed-form solutions. For real molecules, potentials are complicated, electrons interact, and analytic expressions are rarely obtainable. In practice, chemists often rely on numerical methods to discover physically meaningful solutions rather than writing them down algebraically.

To see how this works in a controlled setting, we will approach the particle-in-a-box problem *numerically*, instead of *analytically* (like you may have in lecture). 

Instead of solving for the eigenfunction directly, we define a quantity that measures how "energetically costly" a candidate wavefunction is and minimize it. One useful choice is the average value (expectation value) of the energy, which is well-defined even for non-eigenfunctions:
$$
\braket{E} = \frac{\int_0^L \psi^*(x)\hat{H}\psi(x) dx}{\int_0^L |\psi(x)|^2 dx}.
$$
Where $\hat{H} = \hat{T} =  -\frac{\hbar{}^2}{2m}\frac{d^2}{dx^2}$ for the particle-in-a-box system.

The numerator measures energy and the denominator ensures normalization. Minimizing E is equivalent to asking: Among all functions that satisfy the boundary conditions, which one has the lowest kinetic energy?

---

### Coding Activity 1
`25 points`

- C4.1: Solve the Schrödinger equation for an electron using numerical methods.
- P4.1: Perform numerical optimization routines with `scipy`.



#### Part A - Calculate Energies For Trial Wavefunctions

Write at least three different trial functions for $\psi(x)$ that satisfy the boundary conditions $\psi(0) = \psi(L) = 0$. Some ideas:
- A polynomial like $x(L - x)$
- A Gaussian multiplied by $x(L - x)$
- A sine wave that satisfies the boundary conditions like $sin(\frac{\pi{}}{L}x)$

Each function should take an array `x` as input and return an array of the same shape.

For each trial function:
1. Plot it using the `plot_wavefunction` function defined below.
2. Compute its energy using `compute_energy` from the helper.
3. Record which function has the lowest energy.

In [ ]:
# --- Given: plotting utility ---
# This function plots a trial wavefunction on a box of length L.
# You can read through this to see how it works, but you don't need to modify it.
def plot_wavefunction(x, y, label=None):
    """Plot a trial wavefunction and shade the box region."""
    if not abs(y[0]) < 0.0001 or not abs(y[-1]) < 0.0001:
        raise ValueError("Trial wavefunction must satisfy boundary conditions (psi(0) = psi(L) = 0).")
    # evaluate energy
    E = compute_energy(y, x)
    plt.plot(x, y, label=label + f" (E={E:.3f})")
    plt.axhline(0, color='grey', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.4)
    plt.axvline(x[-1], color='black', linewidth=1, linestyle='--', alpha=0.4)
    plt.xlabel('x')
    plt.ylabel(r'$\psi(x)$')
    plt.title('Wavefunctions & Energies')
    plt.legend()

In [ ]:
L = 2.0
x = np.linspace(0, L, 200)

# Example: Gaussian bump (provided)
def gaussian(x):
    center, width = L/2, L/5 
    return np.exp(-((x - center)**2) / width**2) * (x * (L - x))  # Ensure it goes to zero at boundaries


plot_wavefunction(x, gaussian(x), label='Gaussian')

In [ ]:
# YOUR CODE HERE — define another trial function (e.g. a sine wave, sin(pi x / L)) and plot it

In [ ]:
# YOUR CODE HERE - define another trial function and plot it

#### Part B - Optimizing wavefunctions

You've constructed several guess wavefunctions and evaluated their energies. Imagine you wanted to find the wavefunction which had the lowest energy. How would you approach this?

Rather than try different functional forms by hand, we can just use our energy functional and our guess wavefunctions to **optimize** the wavefunction! 

We will use a numerical optimization algorithm which will iteratively refine our guess to minimize the energy until it cannot make the energy any lower. 

**Your task:**

- Use `scipy.optimize.minimize` to find the wavefunction that minimizes the energy.
- Plot the wavefunction before and after minimization using plot_wavefunction().
- **Leave comments explaining the worked example.**

(Hint: It will help you to write down the names of the functions you created as a reference for this part)

In [ ]:
# Subgoal: Minimize energy for the Gaussian trial function

y_guess = gaussian(x)                                                       #YOUR COMMENT HERE
result_gauss = minimize(compute_energy, y_guess, args=(x,), method='CG')    #YOUR COMMENT HERE

#the variable name 'result_gauss.x' is confusing here. it is really the y values in our case, 
# but this is the convention minimize uses.
phi_gauss_opt = result_gauss.x                                              #YOUR COMMENT HERE
plot_wavefunction(x, y_guess, label='Gaussian (initial)')                   #YOUR COMMENT HERE
plot_wavefunction(x, phi_gauss_opt, label='Gaussian (optimized)')           #YOUR COMMENT HERE
plt.show()

In [ ]:
# YOUR CODE HERE - repeat for your second trial function

In [ ]:
# YOUR CODE HERE - repeat for your third trial function

### Question 1
`15 points`

- C4.1: Solve the Schrödinger equation for an electron using numerical methods.

a) Did the converged result of minimization depend on your initial guess? If so, why do you think this is?

b) Were there any functions for which the energy (and function) remained unchanged after optimization? If so, what does this tell you about that function?

c) Explain how this technique might be useful for real chemical systems where no analytic solution exists.

---

*Your answer here (`double click me!`):*

<br></br>

<a id="part2"></a>

## Part 2 - Energy Levels of the Particle in a Box

**Numerical vs. Analytic**

In Part 1, we used numerical minimization to find the ground-state wavefunction, the lowest-energy solution. This was the state the optimizer converged to, and in our case it was a sine function. 

Our optimizer actually gave us the expected analytic result! Now, we know from the analytic solutions of the Schrodinger Equation for the particle-in-a-box that multiple solutions exist, but we have only found one so far with our optimization technique.

**Next Steps**

This raises an interesting question - are there other wavefunctions our optimizer will converge at? If so, what are their energies? Furthermore, how does the energy change as we change length? 

These are all questions we can answer by using different trial wavefunctions, optimizing them, and numerically evaluating their energies. Our approach will be first to examine a few representative cases to gain intuition, then to generate large amounts of data and fit a function to it. 

---

### Coding Activity 2
`20 points`

- C4.2: Explain features of particle-in-a-box wavefunctions and their energies.
- P4.2: Fit nonlinear curves to data using numerical optimziation.

#### Part A - Excited states

In Part 1, you saw that the optimizer finds the ground-state wavefunction by minimizing the energy. We found that $\psi_n(x) = \sin(\pi x / L)$ was the ground state wavefunction. 

If there are other energy minima, we might expect them to be sine waves, like our ground state was. 

**Your task:**

- To explore this, construct trial wavefunctions $\psi_n = \sin(n\pi x/ L)$ for $n = 1, 2, 3$ at a fixed box length, optimize each one, and check whether the optimizer changes them. 

- Use `plot_wavefunction` to compare before and after.

- Leave comments on the worked example explaining what each line does.

In [ ]:
# n = 1 (worked example)
L = 4.0                             # YOUR COMMENT HERE
x = np.linspace(0, L, 200)          # YOUR COMMENT HERE

psi_1 = np.sin(1 * np.pi * x / L)   # YOUR COMMENT HERE 

# we can split up the arguments of a function into multiple lines for readability
result_1 = minimize(                # YOUR COMMENT HERE
    compute_energy,                 
    psi_1,                            
    args=(x,),                       
    method='CG'
    )

plot_wavefunction(x, psi_1, label='n=1 (before)') # YOUR COMMENT HERE
plot_wavefunction(x, result_1.x, label='n=1 (after)')
plt.title('n = 1: Before and After Optimization')
plt.show()

In [ ]:
# YOUR CODE HERE — repeat for n = 2


In [ ]:
# YOUR CODE HERE — repeat for n = 3


#### Part B - Varying the box length

**Your task:**

- Now fix $n = 1$ (the ground state) and repeat the same process for several different box lengths. 
- Construct $\psi_1(x) = \sin(\pi x / L)$ for each $L$, optimize, and confirm that each one is unchanged.

In [ ]:
# YOUR CODE HERE - repeat for L = 4.0, n = 1
# HINT: you can use the code from the n=1 worked example here!

# add this line to your plotting instructions before you call plt.plot()
plt.xlim(0,12) # makes sure axis limits are the same for all plots so we can compare them

In [ ]:
# YOUR CODE HERE - repeat for L = 8.0, n = 1

In [ ]:
# YOUR CODE HERE - repeat for L = 12.0, n = 1

#### Part C - Discovering the energy formula

You've seen how $E$ depends on $n$ and $L$ individually. Now we're going to try to find the function that maps n and L to the energy E.

We'll generate data points using a `for` loop and then use `scipy.optimize.curve_fit` to optimize a function to fit it, like we did in the warmup.  

**Your task:** 

- The model function `energy_model` does not fit the data. Modify it until the surface matches the data points.
**Add comments to the provided code explaining what each section does.**

In [ ]:
# --- Provided: compute energies for all (n, L) combinations ---
n_values = [1, 2, 3, 4, 5, 6]           # YOUR COMMENT HERE
L_values = [4, 6, 8, 10, 12]

n_list, L_list, E_list = [], [], []     # YOUR COMMENT HERE

for n in n_values:                      # YOUR COMMENT HERE
    for L in L_values:
        x = np.linspace(0, L, 500)      # YOUR COMMENT HERE
        psi = np.sin(n * np.pi * x / L) # YOUR COMMENT HERE
        E = compute_energy(psi, x)      # YOUR COMMENT HERE
        n_list.append(n)                # YOUR COMMENT HERE
        L_list.append(L)
        E_list.append(E)

n_arr = np.array(n_list)                # YOUR COMMENT HERE
L_arr = np.array(L_list)
E_arr = np.array(E_list)

In [ ]:
# Modify this function until the model surface matches the data points!
# curve_fit will optimize a, b, c for you — you just need the right functional form.
# If curve_fit fails, try adjusting p0 (the initial guess for the parameters).

def energy_model(nL, a, b):
    n, L = nL
    return a * n / (L) + b  # this function doesn't fit the data! fix it!

from scipy.optimize import curve_fit
popt, pcov = curve_fit(
    energy_model, 
    (n_arr, L_arr),
    E_arr, 
#    p0=[1.0, 1.0] # uncomment this to provide initial guess parameters if needed
    )
print(f"Fitted parameters:")
print(popt)

In [ ]:
# --- Provided: plot data points and model surface ---
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Numerical data as scatter points
ax.scatter(L_arr, n_arr, E_arr, c='red', s=50, label='Numerical data')

# Model surface
n_grid, L_grid = np.meshgrid(np.linspace(1, 6, 30), np.linspace(4, 12, 30))
E_model = energy_model((n_grid, L_grid), *popt)
ax.plot_surface(L_grid, n_grid, E_model, alpha=0.3, cmap='viridis')

ax.set_xlabel('Box length L (a.u.)')
ax.set_ylabel('Quantum number n')
ax.set_zlabel('Energy (a.u.)')
ax.set_title('PIB Energy: Data vs. Model')
ax.legend()
plt.tight_layout()
plt.show()

### Question 2
`10 points`

- C4.2: Explain features of particle-in-a-box wavefunctions and their energies.

a) How does the shape of the wavefunction change as we increase n? As we increase L?

b) How does the energy of the wavefunction depend on the number of nodes? 

c) State the expression which best fit your data. Does it physically make sense in light of your answers to (a) and (b)? Explain your reasoning. 

---

*Your answer here (`double click me!`):*

<br></br>

<a id="part3"></a>

## Part 3 - Computational Chemistry

So far we have worked with the particle-in-a-box as a simplified model. Now we use quantum chemistry software to compute the electronic structure of real molecules and see how the model compares.

**The Self-Consistent Field Procedure**


We use the Hartree–Fock / Self-Consistent Field (SCF) method. This method treats the energy of each electron as being a function of its kinetic energy, interaction with the nuclei, and interaction with other electrons.

$$\epsilon_{i}\psi_i(r) = -\frac{\hbar{}^2}{2m}\nabla^2\psi_i(r) + V_{Ne}\psi_i(r) + V_{ee}(r)\psi_i(r)$$
Where $\epsilon_{i}$ is the energy of orbital $\psi_i$, and the terms on the right represent the kinetic energy of the electron, the potential energy from electron-nuclear interactions, and the potential energy from electron-electron interactions, respectively.

The last one is the really hard part, and it's what makes molecular wavefunctions impossible to solve analytically in general! The Hartree-Fock procedure handles this by treating electrons as interacting only with the *average* density of the other electrons.

Because the potential energy of every electron depends on the positions of every other electron, we can't solve it all at once, and so we must optimize it iteratively until it converges to a *self-consistent* solution.


**The application**

Conjugated polyenes are a chemical system that resembles the particle-in-a-box model quite closely. The spacings of their frontier orbital energy levels correspond to the wavelengths of light that they will absorb (because light promotes electrons to excited states). 

Right now, we're interested in how the PIB model predicts the $\pi$ orbital energies of octatetraene, as calculated with the SCF method. How close do you think they'll be?

---

### Coding Activity 3
`30 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- P4.3: Perform quantum chemical calculations with `psi4`.

#### Part A - Running a Hartree–Fock calculation

It's time for us to become real computational chemists! We will calculate the orbital energies of an extended alkene with the self-consistent field technique. How well do you think a real molecule can be modeled by as simple a model as the particle-in-a-box?

**Your task** 

- The cell below runs an HF/STO-3G calculation on ethane and plots its orbital energies. Run it and see what happens!

- Then, in the next cell, run the same calculation on **octatetraene** (`data/octatetraene.xyz`) and plot its orbital energies, based on the provided example.

In [ ]:
# --- Worked example: ethane ---
psi4.set_output_file('output.dat', False)
psi4.set_memory('1 GB')

mol_ethane = load_psi4_molecule('data/ethane.xyz') 
energy_ethane, wfn_ethane = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=mol_ethane)
print(f'Ethane HF/STO-3G energy: {energy_ethane:.6f} a.u.')

# Plot all orbital energies
eps_ethane = wfn_ethane.epsilon_a().np
plt.plot(eps_ethane, 'o', markersize=5, color='steelblue')
plt.axhline(0, color='grey', linewidth=0.5)
plt.xlabel('Orbital index')
plt.ylabel('Energy (a.u.)')
plt.title('Ethane: All HF/STO-3G Orbital Energies')
plt.show()

In [ ]:
# YOUR CODE HERE
# Run HF/STO-3G on octatetraene ('data/octatetraene.xyz')
# Store the wavefunction in wfn_oct — you will need it in Parts B and C
# Plot all orbital energies


#### Part B - Visualizing molecular orbitals

The cell below generates cube files for orbitals near the HOMO and LUMO of octatetraene and displays them with `fortecubeview`. Run it, then examine the orbital shapes.

Octatetraene ($\text{C}_8\text{H}_{10}$) has $N_e = 58$ electrons, so the HOMO is orbital index 29. 

**Note** - Fortecubeview uses 1-based indexing while Python uses 0-based indexing. You'll need to subtract 1 from your orbital indices to use them in your code.

**Your task:**

- Identify which of the displayed orbitals are $\pi$ orbitals - they have lobes above and below the molecular plane. Record their indices; you will use them in Part C.

In [ ]:
# --- Given: orbital visualization ---
import os
os.makedirs('cubes', exist_ok=True)

# psi4 CUBEPROP_ORBITALS uses 1-indexed orbital numbers
homo_psi4 = 29  # octatetraene HOMO (1-indexed; 58 electrons / 2 = 29)
orb_numbers = list(range(homo_psi4 - 4, homo_psi4 + 5))  # [25, 26, 27, 28, 29, 30, 31, 32, 33]

psi4.set_options({
    'CUBEPROP_TASKS': ['orbitals'],
    'CUBEPROP_FILEPATH': 'cubes',
    'CUBEPROP_ORBITALS': orb_numbers,
})
psi4.cubeprop(wfn_oct)
fortecubeview.plot('cubes', colorscheme='wow')

In [ ]:
# Run this cell when you are done inspecting orbitals
shutil.rmtree('cubes', ignore_errors=True)

In [ ]:
# Record the orbital indices that you identified as pi orbitals:
pi_indices = np.array([]) - 1  # YOUR CODE HERE — e.g. [25, 26, 27, 28, 29, 30, 31] 
# we subtract 1 to convert between 1- and 0-based indexing

#### Part C - Fitting the PIB model to $\pi$ orbital energies

Now for the payoff - let's see how the $\pi$ orbitals specifically compare to the PIB energies. We shouldn't expect the absolute values to be the same, of course - so we'll use the basic functional form of the PIB equation while allowing the proportionality constant and intercept to vary.

**Your task:**

Plot only the $\pi$ orbital energies, then try to fit the particle-in-a-box model $E = A \cdot n^2 + b$ to them using `curve_fit`. Does the PIB model capture the pattern?

In [ ]:
# GIVEN CODE
# Subgoal: Plot pi orbital energies
AU_TO_EV = 27.211
eps = wfn_oct.epsilon_a().np
pi_energies = eps[pi_indices] * AU_TO_EV
n_pi = np.arange(1, len(pi_energies) + 1)
# Plot pi energies
plt.plot(n_pi, pi_energies, 'o-', color='darkorange')
plt.xlabel(r'$\pi$ orbital number')
plt.ylabel('Energy (eV)')
plt.title(r'Octatetraene: $\pi$ Orbital Energies')
plt.show()

In [ ]:
# Fit E = A * n^2 + b


# define the model function
# should accept n, A, b as arguments and return E
def pib_model  #FILL IN THE REST

# use curve_fit to fit the model to the data (store result in popt, _)
# use the examples from earlier in the notebook as a guide!

In [ ]:
# GIVEN CODE
# Plot data vs fit
n_fine = np.linspace(1, len(pi_energies), 50)
plt.plot(n_pi, pi_energies, 'o', label='HF data')
plt.plot(n_fine, pib_model(n_fine, *popt), '--', label=f'PIB fit: E = {popt[0]:.3f}$n^2$ + {popt[1]:.3f}')
plt.xlabel(r'$\pi$ orbital number')
plt.ylabel('Energy (eV)')
plt.title('PIB Model vs. HF Orbital Energies')
plt.legend()
plt.show()

### Question 3
`15 points`

- C4.2: Explain features of particle-in-a-box wavefunctions and their energies.
- C4.4: Evaluate the tradeoffs of different physical models.

a) Look at the $\pi$ orbitals of octatetraene (which you visualized in part B). In what ways do they resemble the particle-in-a-box wavefunctions from Part 2?

b) How well does the PIB model $E = An^2 + b$ fit the $\pi$ orbital energies? What might account for any discrepancies?

c) Compare the Hartree–Fock procedure to the variational minimization you did in Part 1. What is conceptually the same? What is different?

---

*Your answer here (`double click me!`):*

<br></br>

<a id="part4"></a>

## Part 4 - Comparing Models

**The philosophy**

We now have two models to compare - one very crude and one very sophisticated. We may be tempted to call the sophisticated model "better," but as the proverb goes,
> All models are wrong, but some are useful.

With all models, there is a tradeoff between conceptual clarity (and computational expense!) and accuracy. The best model gets the most for the least, and there's a real art to this. 

The particle-in-a-box model is ridiculously spartan compared to the complexity of real molecules. So how far will it get us?

**The application**

We're going to predict the principal absorption energy, $E = \frac{hc}{\lambda_{max}}$ for conjugated polyenes using these two models, and compare to experiment. 

Each orbital holds two electrons (spin-up and spin-down). The highest filled orbital is the HOMO and the lowest empty orbital is the LUMO. Roughly speaking, their energy difference

$$\Delta E = E_{\text{LUMO}} - E_{\text{HOMO}}$$

determines the energy of light the molecule can absorb. In Part 4 we will compare these gaps to the particle-in-a-box prediction, and to the prediction of the more sophisticated SCF method.

---

### Coding Activity 4
`20 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- C4.4: Evaluate the tradeoffs of different physical models.
- P4.2: Fit nonlinear curves to data.

#### Part A - HF HOMO–LUMO Gaps

The cell below runs a Hartree–Fock calculation on **butadiene** and extracts the HOMO–LUMO gap. 

- **Leave comments on the worked example explaining what this does** 

In [ ]:
# --- Worked example: butadiene HOMO-LUMO gap ---
mol_butadiene = load_psi4_molecule('data/butadiene.xyz') # YOUR COMMENT HERE
energy_butadiene, wfn_butadiene = psi4.energy('SCF/STO-3G', return_wfn=True, molecule=mol_butadiene)

num_carbons = 4

# this gives us the orbital energies of our molecule as a numpy array
eps_butadiene = wfn_butadiene.epsilon_a().np         # YOUR COMMENT HERE

n_e_butadiene = num_carbons * 7 + 2                  # YOUR COMMENT HERE
# why do we divide by 2?
# why do we subtract 1?
homo_idx = n_e_butadiene // 2 - 1                    # YOUR COMMENT HERE
lumo_idx = homo_idx + 1                              # YOUR COMMENT HERE
gap_butadiene = (eps_butadiene[lumo_idx] - eps_butadiene[homo_idx]) # YOUR COMMENT HERE
gap_butadiene *= 27.211  # convert to eV

print(f'Butadiene HF/STO-3G energy: {energy_butadiene:.6f} a.u.')
print(f'HOMO index: {homo_idx}, LUMO index: {lumo_idx}')
print(f'HOMO-LUMO gap: {gap_butadiene:.2f} eV')

In [ ]:
# Subgoal: Compute HF HOMO-LUMO gaps for the polyenes

polyene_nc = [4, 6, 8, 10, 12, 14, 16]
polyene_xyz = [
    'data/butadiene.xyz',
    'data/hexatriene.xyz',
    'data/octatetraene.xyz',
    'data/decapentaene.xyz',
    'data/dodecahexaene.xyz',
    'data/tetradecaheptaene.xyz',
    'data/hexadecaoctaene.xyz',
]

hf_gaps = []

# YOUR CODE HERE
# Loop over polyene_nc and polyene_xyz

# use the worked example code inside the for loop!
# generalize!
for n, xyz_file in zip(polyene_nc,polyene_xyz):
    print(n)
    print(xyz_file)
    # now simply use the code above, but genericized!
    #   1. Load the psi4 molecule with load_psi4_molecule(xyz_file)
    #   2. Run psi4.energy('SCF/STO-3G', ...)
    #   3. Get orbital energies with wfn.epsilon_a().np
    #   4. Compute n_e = 7 * n_c + 2, homo_idx = n_e // 2 - 1
    #   5. Compute gap in eV and append to hf_gaps

print('HF HOMO-LUMO gaps (eV):', hf_gaps)

In [ ]:
# Plot HF gaps vs chain length

plt.figure(figsize=(8, 5))
plt.plot(polyene_nc, hf_gaps, 's-', color='steelblue', label='HF/STO-3G')
plt.xlabel('Number of carbon atoms')
plt.ylabel('HOMO-LUMO gap (eV)')
plt.title('HF HOMO-LUMO Gaps for Conjugated Polyenes')
plt.legend()
plt.show()

#### Part B - PIB HOMO–LUMO Gaps

Earlier, you found an equation which gives the PIB energy as:

$$E_n = \frac{n^2 \pi^2}{2 L^2} \quad \text{(atomic units)}$$

(check that your numerical constant ~4.9 is actually $\pi^2/2$ if you would like!)

We're going to use this expression to make a model for polyene HOMO-LUMO gaps which, in the crudest way possible, accounts for the competing effects of increasing length and increasing number of nodes in the frontier orbitals. But will it fit?

Some useful information:
- 2 electrons to a molecular orbital, 1 electron to a PIB energy level
- For a polyene with $n_c$ carbon atoms:
    - Box length: $L = 1.4 \times (n_c - 1)$ Å, converted to a.u. by multiplying by 1.8897
    - HOMO quantum number: $n_{\text{HOMO}} = n_c / 2$ 
    - LUMO quantum number: $n_{\text{LUMO}} = n_c / 2 + 1$
    - Gap: $\Delta E = E_{\text{LUMO}} - E_{\text{HOMO}}$, converted to eV by multiplying by 27.211

**Your task:**
- Translate the provided psuedocode into a usable function and plot its results.

In [ ]:
def pib_energy_gap(n_c):
    """
    Compute the PIB HOMO-LUMO gap for a polyene with n_c carbons.
    
    Parameters
    n_c : (int) Number of carbon atoms.
    
    Returns
    gap_eV : (float) HOMO-LUMO gap in eV.
    """
    # YOUR CODE HERE
    # 1. Compute L in Angstroms: L = 1.4 * (n_c - 1)
    # 2. Convert to atomic units: L_au = L * 1.8897
    # 3. n_homo = n_c // 2, n_lumo = n_c // 2 + 1
    # 4. E_n = n^2 * pi^2 / (2 * L_au^2)
    # 5. gap = (E_lumo - E_homo) * 27.211
    # 6. make sure we return the result!

In [ ]:
# Subgoal: Compute PIB gaps and plot alongside HF gaps

pib_gaps = []

# YOUR CODE HERE
for n in polyene_nc:
# 1. Loop over polyene_nc and compute pib_energy_gap for each
# 2. Plot both hf_gaps and pib_gaps vs polyene_nc on the same axes
#    with labels, legend, axis labels, and title


In [ ]:
# plot your results!
plt.figure(figsize=(8, 5))
plt.plot(polyene_nc, pib_gaps, 'o-', color='darkorange', label='PIB')
plt.xlabel('Number of carbon atoms')
plt.ylabel('HOMO-LUMO gap (eV)')
plt.title('HF vs PIB HOMO-LUMO Gaps')
plt.legend()
plt.show()

#### Part C - Comparison to Experiment

We have experimental UV absorption data for these polyenes. Let's see how both models compare to reality.

We'll also apply a linear correction to each model:

$$E_{\text{corrected}} = \alpha \cdot E_{\text{model}} + \beta$$

This fits two parameters ($\alpha$, $\beta$) to best match the experimental data. If the model has the right shape (even if absolute values disagree), the correction should give good agreement.

**Your task:**
- Plot the PIB and SCF predictions versus experimental results.
- Apply a linear correction to each model.
- Plot the corrected predictions versus experimental results.

In [ ]:
# --- Given: load experimental data ---
expt_data = pd.read_csv('data/polyene_excitation_energies.csv')
print(expt_data)

expt_energies = expt_data['energy_eV'].values

In [ ]:
# Subgoal: Plot all three datasets (HF, PIB, experiment) on the same axes

plt.plot(polyene_nc, hf_gaps, 's-', label='HF/STO-3G')
plt.plot(polyene_nc, pib_gaps, 'o-',  label='PIB')
plt.plot(polyene_nc, expt_energies, 'D-', label='Experiment')
plt.xlabel('Number of carbon atoms')
plt.ylabel('Excitation energy (eV)')
plt.title('Model vs Experimental Excitation Energies')
plt.legend()
plt.show()

In [ ]:
# Subgoal: Fit linear corrections to both models

def linear_model(x, alpha, beta):
    """Linear correction: alpha * x + beta."""
    return alpha * x + beta

# Subgoal: Fit HF model to experiment
popt_hf, _ = curve_fit(linear_model, hf_gaps, expt_energies)
print(f'HF correction:  alpha = {popt_hf[0]:.4f}, beta = {popt_hf[1]:.4f}')

# YOUR CODE HERE --- Fit PIB model to experiment and print parameters


In [ ]:
# Subgoal: Compare corrected predictions to experiment

# 1. Compute corrected predictions:
hf_corrected = linear_model(np.array(hf_gaps), *popt_hf) # * is the unpacking operator if used on a list
# likewise for PIB!

# 2. Compute and print R^2 for each:
print(calculate_r_squared(expt_energies, hf_corrected))
# likewise for PIB!

# 3. Plot expt_energies, hf_corrected, pib_corrected vs polyene_nc
#    copy/paste and modify your plotting code from a couple cells ago to plot the corrected versions


### Question 4
`10 points`

- C4.3: Apply the particle-in-a-box model to conjugated polyenes.
- C4.4: Evaluate the tradeoffs of different physical models.

a) How well do the SCF and PIB models fit the experimental data before and after the linear correction? Which is the more accurate method?

b) Look at the timings at the bottom of the notebook cells where you ran your calculations. How long did the SCF calculations take? How about the PIB calculations?

c) Evaluate the relative utility of the PIB and SCF models in terms of accuracy, computational cost, and conceptual clarity. Give an example of a scenario in which you would use each.

---

*Your answer here (`double click me!`):*

<br></br>

<a id="reflection"></a>

## Reflection
`10 points`

a) How did your experience using numerical techniques compare to your experience solving equations analytically (as in the lecture)? Which do you prefer and why?

b) How could you know whether an approximate model was appropriate for a physical system you wanted to study? Give at least two ways you could approach this and explain your reasoning. 

c) When completing this lab, what did you feel most and least confident about and why? What are some factors that have been helping or hurting your learning?

*Your answer here (`double click me!`):*

<br></br>

## References
1. Excitation energy data: 
    
    Chauhan, J. S.; Patel, R. P. Interpretation & Studies of the Visible Spectra and to Exploit Properties of Polyenes and to Develop a New Method for Classroom Experimental Studies by Use of One Dimensional Box Model. Der Pharma Chemica 2016, 8(5), 67–73.